In [ ]:
from pathlib import Path
import modal
project_name = 'language-model-pretraining'
LOCAL_STORAGE_ROOT = Path('/Users/oguz/Projects/transformer/volume')
app = modal.App(project_name)

container = (
    modal.Image.debian_slim(python_version="3.12") 
    .uv_sync(uv_project_dir="/Users/oguz/Projects/transformer")
    .add_local_python_source("transformer")
)

volume = modal.Volume.from_name('language-model-pretraining', create_if_missing=True)
@app.cls(image=container, volumes={'/storage' : volume})
class Pipeline:
    @modal.enter()
    def setup(self):
        self.storage_root = Path('/storage') if not modal.is_local() else LOCAL_STORAGE_ROOT
        print (self.storage_root)
    @modal.method()
    def show(self):
        for f in (self.storage_root / 'data').iterdir():
            print (f)


# with modal.enable_output():
with app.run():
    Pipeline().show.remote()
with app.run():
    Pipeline().show.local()
